# Regression

It is a supervised machine learning technique used to predict a continuous numerical value based on one or more input features.

## Workflow

**Load → EDA → Preprocessing → Feature Engineering → Train/Test Split → Separate Numerical/Categorical Columns → Create Preprocessing Pipeline → Transform Data → Train Models → Predict → Evaluate**

Load datasets

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/insurance.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
df.describe()

In [ ]:
print("\nData Types:")
print(df.dtypes)

In [ ]:
print("\nUnique Values:")
for col in df.columns:
    print(col, ":", df[col].nunique())

### Preprocessing

- Handle missing values
- Encode categorical variables
- Handle outliers when justified
- Select useful features
- Feature engineering

For real projects, use `Pipeline` / `ColumnTransformer` to keep preprocessing consistent and avoid leakage.

Separate Traget and Features

In [ ]:
X = df.drop("charges", axis=1)
y = df["charges"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Separate column

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical Columns:")
num_features = num_cols.tolist()
print(num_features)

print("\nCategorical Columns:")
cat_features = cat_cols.tolist()
print(cat_features)

Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Create Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ]
)

Tranform data

In [ ]:
# Linear/Ridge/Lasso/Elastic Net, Polynomial Regression and SVR

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

### Evaluation

Regression metrics:
- **MAE** → lower is better
- **RMSE** → lower is better
- **R²** → higher is better

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": name,
        "MAE": float(round(mae, 2)),
        "RMSE": float(round(rmse, 2)),
        "R2": float(round(r2, 2))
    }

# Linear Regression

**Use:** Simple or approximately linear relationships; good baseline.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_processed, y_train)

y_pred_lr = model.predict(X_test_processed)

result_lr = evaluate("Linear Regression", y_test, y_pred_lr)
result_lr

# Polynomial Regression

**Use:** When the relationship is curved/non-linear but can be represented with polynomial features.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

model.fit(X_train_processed, y_train)

y_pred_poly = model.predict(X_test_processed)

result_poly = evaluate("Polynomial Regression", y_test, y_pred_poly)
result_poly

# Ridge Regression

**Use:** Linear regression with L2 regularization, useful with correlated features and many predictors.

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)

model.fit(X_train_processed, y_train)

y_pred_ridge = model.predict(X_test_processed)

result_ridge = evaluate("Ridge", y_test, y_pred_ridge)
result_ridge

# Lasso Regression

**Use:** Linear regression with L1 regularization; useful when feature selection is helpful.

In [ ]:
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.01, max_iter=10000)

model.fit(X_train_processed, y_train)

y_pred_lasso = model.predict(X_test_processed)

result_lasso = evaluate("Lasso", y_test, y_pred_lasso)
result_lasso

# Elastic Net

**Use:** Combines L1 and L2 regularization when you want both regularization and feature selection.

In [ ]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(
    alpha=0.01,
    l1_ratio=0.5,
    max_iter=10000
)

model.fit(X_train_processed, y_train)

y_pred_elastic = model.predict(X_test_processed)

result_elastic = evaluate("Elastic Net", y_test, y_pred_elastic)
result_elastic

# Decision Tree Regressor

**Use:** Non-linear tabular data when rule-based splits are useful. Scaling is not required.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred_tree = model.predict(X_test_processed)

result_tree = evaluate("Decision Tree", y_test, y_pred_tree)
result_tree

# Random Forest Regressor

**Use:** Strong general-purpose model for non-linear tabular data. Scaling is not required.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

y_pred_rf = model.predict(X_test_processed)

result_rf = evaluate("Random Forest", y_test, y_pred_rf)
result_rf

# Gradient Boosting Regressor

**Use:** Strong tabular regression where sequential trees can improve previous errors.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred_gb = model.predict(X_test_processed)

result_gb = evaluate("Gradient Boosting", y_test, y_pred_gb)
result_gb

# XGBoost Regressor

**Use:** High-performance gradient boosting for many tabular regression problems.

In [ ]:
# pip install xgboost

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred_xgb = model.predict(X_test_processed)

result_xgb = evaluate("XGBoost", y_test, y_pred_xgb)
result_xgb

# LightGBM Regressor

**Use:** Fast gradient boosting, especially useful for larger tabular datasets.

In [ ]:
# pip install lightgbm

from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

model.fit(X_train_processed, y_train)

y_pred_lgbm = model.predict(X_test_processed)

result_lgbm = evaluate("LightGBM", y_test, y_pred_lgbm)
result_lgbm

# CatBoost Regressor

**Use:** Gradient boosting that is convenient when categorical features are important.

In [ ]:
# pip install catboost

from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=200,
    learning_rate=0.05,
    depth=6,
    verbose=False,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

y_pred_cat = model.predict(X_test)

result_cat = evaluate("CatBoost", y_test, y_pred_cat)
result_cat

# SVR

**Use:** Small/medium datasets with complex relationships; scaling is important.

In [ ]:
from sklearn.svm import SVR

model = SVR(
    kernel="rbf",
    C=10,
    epsilon=0.1
)

model.fit(X_train_processed, y_train)

y_pred_svr = model.predict(X_test_processed)

result_svr = evaluate("SVR", y_test, y_pred_svr)
result_svr

# Model Comparison

- **MAE:** lower is better
- **RMSE:** lower is better
- **R²:** higher is better

Compare all models using the same test set.

In [ ]:
results = pd.DataFrame([
    result_lr,
    result_poly,
    result_ridge,
    result_lasso,
    result_elastic,
    result_tree,
    result_rf,
    result_gb,
    result_xgb,
    result_lgbm,
    result_cat,
    result_svr
])

results.sort_values("RMSE").reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

comparison = results.sort_values("RMSE")

plt.figure(figsize=(10, 6))
plt.barh(comparison["Model"], comparison["RMSE"])
plt.xlabel("RMSE (lower is better)")
plt.title("Regression Model Comparison")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

comparison = results.sort_values("R2")

plt.figure(figsize=(10, 6))
plt.barh(comparison["Model"], comparison["R2"])
plt.xlabel("R² Score (higher is better)")
plt.title("Regression Model Comparison")
plt.gca().invert_yaxis()
plt.show()

## Model Improvement

**Baseline → Feature Engineering → Compare Models → Select promising model → Hyperparameter Tuning → Cross-Validation → Final Model → Final Test Evaluation**

### Feature Engineering
Create, transform or select useful features.

### Hyperparameter Tuning
Change model settings such as:
- `n_estimators`
- `max_depth`
- `learning_rate`
- `alpha`
- `C`

### Cross-Validation
Evaluate a model across multiple training/validation splits instead of trusting one split.

### GridSearchCV
Tests all specified hyperparameter combinations.

### RandomizedSearchCV
Tests a selected number of parameter combinations.

In [ ]:
# RandomizedSearchCV for Random Forest

from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

params = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", "log2"]
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=params,
    n_iter=20,
    cv=5,  # Cross-validation
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_processed, y_train)

print("Best parameters:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

In [ ]:
best_model = search.best_estimator_

y_pred_final = best_model.predict(X_test_processed)

final_result = evaluate(
    "Tuned Random Forest",
    y_test,
    y_pred_final
)

final_result

# Notes

**Linear Regression** → simple linear relationship

**Polynomial Regression** → curved relationship

**Ridge** → linear + L2 regularization

**Lasso** → linear + L1 regularization / feature selection

**Elastic Net** → L1 + L2

**Decision Tree** → rule-based non-linear model

**Random Forest** → many decision trees

**Gradient Boosting** → sequential trees correcting errors

**XGBoost** → optimized gradient boosting

**LightGBM** → fast gradient boosting

**CatBoost** → boosting with strong categorical-data support

**SVR** → margin-based regression; scaling important